In [34]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
sys.path.append("..")
from src.io_utils import load_data
df = load_data("../data/health_study_dataset.csv")

**1. Scatterplotet (spridningsdiagram) hjälper oss att undersöka:**
- Verkar blodtrycket öka med åldern?
- Finns det något samband?
- Hur mycket varierar blodtrycket mellan personer i samma ålder?
- Finns det några avvikande värden?
- OBS: scatterplotet räknar inte ut någon modell. Det visar bara våra observationer.

**2. Linjär regression – beräkna ett samband**
- Nu har vi lagt till en röd linje i diagrammet. Det är en regressionslinje som representerar den linje som bäst sammanfattar det linjära sambandet mellan ålder och blodtryck enligt minsta kvadratmetoden.
- Med den kan vi exempelvis uppskatta: vilket systoliskt blodtryck modellen förväntar sig vid en viss ålder.
- Vår linjära regression kan uppskatta blodtrycket även vid åldrar där vi saknar observationer, eftersom modellen utgår från den övergripande trenden i hela datasetet

**3. Men var kommer signifikans in?**
Här är en viktig skillnad:
- Linjär regression: Vi undersöker och modellerar sambandet mellan två variabler.
- Signifikansanalys: Vi kan undersöka om det finns statistiskt stöd för att sambandet skiljer sig från noll i populationen.
- Vi vill undersöka om lutningen i regressionslinjen är tillräckligt tydlig för att det ska vara osannolikt att den uppstått enbart på grund av slumpmässig variation

### Utökad analys och visualisering
- Lägg till minst en ny analys eller graf som ger en djupare förståelse (t.ex. relationen mellan blodtryck och ålder, eller sjukdomsförekomst per kön).

#### - Nollhypotes (H₀): Det finns inget linjärt samband mellan ålder och blodtryck (lutningen är 0)
#### - Alternativ hypotes (H₁): Det finns ett linjärt samband (lutningen skiljer sig från 0)

In [35]:
# MULTIPEL REGRESSIONSANALYS

import statsmodels.api as sm
np.random.seed(42)

# # # Source - https://stackoverflow.com/a/40902719
# # # Posted by piRSquared
# # # Retrieved 2026-11-20, License - CC BY-SA 3.0

# (Y) beroende variabel: systolic_bp
# (X) oberoende variabler: age, weight, cholesterol, smoking, disease, sex

# KOMMENTAR
# samtliga oberoende variabler är numeriska utom sex (M/F) och smoking (Yes/No)
# dessa två behöver konverteras från strings till numeriska variabler (1/0)

c_smoker = df["smoker"].map({"Yes": 1, "No": 0}) # omvandlar (convert) smoker från Yes/No till 1/0
c_sex = df["sex"].map({"M": 1, "F": 0})
df_drop = df.drop(columns=["id", "smoker","sex"])            # tar bort originalkolumnen
df_assign = df_drop.assign(smoker=c_smoker, sex=c_sex)      # lägger till den omkodade kolumnen
df_processed = df_assign

# plockar ut variablerna til X och Y
# oberoende variabler
X = df_processed[["age", "height", "weight", "cholesterol", "disease", "smoker", "sex"]]

# test..
X = df_processed[["age"]]
# beroende variabel
Y = df_processed[["systolic_bp"]]

# model = sm.OLS(Y, X)
# results = model.fit()
# print(results.summary())



In [ ]:
# MULTIPEL REGRESSIONSANALYS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
np.random.seed(42)

# Källa: https://www.youtube.com/watch?v=KYNuzfn5Fx0
# Source - https://stackoverflow.com/a/40902719
# Posted by piRSquared
# Retrieved 2026-11-20, License - CC BY-SA 3.0

# (Y) beroende variabel: systolic_bp
# (X) oberoende variabler: age, weight, cholesterol, smoking, disease, sex

# KOMMENTAR
# samtliga oberoende variabler är numeriska utom sex (M/F) och smoking (Yes/No)
# dessa två behöver konverteras från strings till numeriska variabler (1/0)

c_smoker = df["smoker"].map({"Yes": 1, "No": 0}) # omvandlar (convert) smoker från Yes/No till 1/0
c_sex = df["sex"].map({"M": 1, "F": 0})
df_drop = df.drop(columns=["id", "smoker","sex"])            # tar bort originalkolumnen
df_assign = df_drop.assign(smoker=c_smoker, sex=c_sex)      # lägger till den omkodade kolumnen
df_processed = df_assign

# plockar ut variablerna til X och Y
# oberoende variabler
X = df_processed[["age", "height", "weight", "cholesterol", "disease", "smoker", "sex"]]

# beroende variabel
Y = df_processed["systolic_bp"]

# lägger till en konstantkolumn (med värdet 1) i X
# konstanten gör att modellen kan beräkna ett grundvärde (intercept)
X_const = sm.add_constant(X)

# sm.OLS() = Ordinary Least Squares (minsta kvadratmetoden)
# Skapar en linjär regressionsmodell som bgiteräknar vilka koefficienter som bäst passar datan
# .fit() = genomför beräkningen och anpassar modellen till datan genom att hitta de bästa koefficienterna
model = sm.OLS(Y, X_const).fit()

# beräknar modellens koefficienter och andra statistiska resultat
# visar bland annat koefficienter, p-värden och förklaringsgrad
print(model.summary())

# TOLKNING
# R-squared: 0.407 = modellen kan statistiskt förklara 
# 40,7 % av variationen i blodtryck mellan personerna.


                            OLS Regression Results                            
Dep. Variable:            systolic_bp   R-squared:                       0.407
Model:                            OLS   Adj. R-squared:                  0.402
Method:                 Least Squares   F-statistic:                     77.73
Date:                fre, 20 nov 2026   Prob (F-statistic):           1.16e-85
Time:                        18:27:47   Log-Likelihood:                -2964.6
No. Observations:                 800   AIC:                             5945.
Df Residuals:                     792   BIC:                             5983.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          99.9181      8.784     11.375      